# 04 — QLoRA: fine-tuning sobre modelo cuantizado a 4-bit

**Level 4 — Model Ops & Optimization**

El mismo LoRA de la section 3, pero el modelo base se carga **cuantizado
a 4-bit** (NF4): ocupa ~6x menos memoria en GPU. Requiere CUDA — sin GPU
el notebook explica qué haría y no entrena.

In [1]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\nQLoRA requiere CUDA: la cuantizacion de bitsandbytes corre en GPU.")
    print("En una maquina sin GPU no se puede entrenar QLoRA; el notebook")
    print("de la section 3 (LoRA normal) funciona en CPU.")

CUDA disponible: True


## Cargar el modelo cuantizado (solo GPU)

In [2]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

    MODELO_BASE = "HuggingFaceTB/SmolLM2-360M-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cuantizacion = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    modelo = AutoModelForCausalLM.from_pretrained(
        MODELO_BASE, quantization_config=cuantizacion
    )
    memoria_mb = torch.cuda.memory_allocated() / 1024 / 1024
    print(f"memoria del modelo en GPU: {memoria_mb:.0f} MB")
    print("(vs ~1450 MB del mismo modelo en fp32: ~6x menos)")

/workspaces/student-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  15%|█▍        | 43/290 [00:00<00:00, 427.96it/s]

Loading weights:  48%|████▊     | 140/290 [00:00<00:00, 743.55it/s]

Loading weights:  82%|████████▏ | 239/290 [00:00<00:00, 851.79it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 822.40it/s]

memoria del modelo en GPU: 245 MB
(vs ~1450 MB del mismo modelo en fp32: ~6x menos)


## QLoRA = LoRA sobre el modelo cuantizado

In [3]:
if torch.cuda.is_available():
    from datasets import Dataset
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
    import json
    from pathlib import Path

    modelo = prepare_model_for_kbit_training(modelo)
    config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    )
    modelo = get_peft_model(modelo, config)
    modelo.print_trainable_parameters()

    ejemplos = [
        json.loads(linea)
        for linea in Path("../data/finetune_dataset.jsonl").read_text(encoding="utf-8").splitlines()
        if linea.strip()
    ]
    textos = [
        tokenizer.apply_chat_template(e["messages"], tokenize=False)
        for e in ejemplos
    ]
    dataset = Dataset.from_dict({"text": textos})

    def tokenizar(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512)

    dataset = dataset.map(tokenizar, batched=True, remove_columns=["text"])

    trainer = Trainer(
        model=modelo,
        args=TrainingArguments(
            output_dir="checkpoints",
            max_steps=40,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=2,
            learning_rate=2e-4,
            logging_steps=10,
            save_strategy="no",
            report_to=[],
            bf16=True,
        ),
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()
    modelo.save_pretrained("../adapters/qlora")
    print("Adaptador QLoRA guardado")

trainable params: 819,200 || all params: 362,640,320 || trainable%: 0.2259


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map: 100%|██████████| 8/8 [00:00<00:00, 1663.42 examples/s]

/workspaces/student-ai/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.693445
20,2.468170
30,2.426407
40,2.260632


Adaptador QLoRA guardado


## Conclusión

- **QLoRA = LoRA + cuantización 4-bit** del modelo base
- La memoria del modelo base baja de ~1.4 GB (fp32) a ~245 MB (NF4)
- Los adaptadores se entrenan igual; la calidad se conserva casi intacta